In [6]:
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Dataset minimal 4 dokumen pendek berbahasa Indonesia[cite: 9]
dokumen_mentah = [
    "Sistem komputer memproses data.",
    "Jaringan komputer menghubungkan sistem.",
    "Kecerdasan buatan dapat meniru manusia.",
    "Sistem temu kembali mencari data."
]

In [7]:
# 2. Preprocessing sederhana (case folding + stopwords removal + cleaning)[cite: 9]
stopwords_sederhana = {"memproses", "menghubungkan", "dapat", "meniru", "mencari"}

dokumen_bersih = []
for doc in dokumen_mentah:
    # Case folding & Cleaning
    doc_clean = re.sub(r'[^a-z\s]', ' ', doc.lower())
    doc_clean = re.sub(r'\s+', ' ', doc_clean).strip()
    
    # Tokenisasi & Stopwords removal
    tokens = [kata for kata in doc_clean.split() if kata not in stopwords_sederhana]
    dokumen_bersih.append(" ".join(tokens))

print("=== HASIL PREPROCESSING ===")
for i, doc in enumerate(dokumen_bersih):
    print(f"D{i+1}: {doc}")

=== HASIL PREPROCESSING ===
D1: sistem komputer data
D2: jaringan komputer sistem
D3: kecerdasan buatan manusia
D4: sistem temu kembali data


In [8]:
# 3. Perhitungan Manual[cite: 9]
print("=== 1. PERHITUNGAN MANUAL TF-IDF ===\n")

# a. Membangun Vocabulary (Kamus Kata)
vocab = set()
for doc in dokumen_bersih:
    vocab.update(doc.split())
vocab = sorted(list(vocab))
N = len(dokumen_bersih)

# b. Menghitung TF (Term Frequency) & DF (Document Frequency)
tf_data = []
df_dict = {term: 0 for term in vocab}

for doc in dokumen_bersih:
    tokens = doc.split()
    tf_row = {term: tokens.count(term) for term in vocab}
    tf_data.append(tf_row)
    for term in set(tokens):
        df_dict[term] += 1

df_tf = pd.DataFrame(tf_data, index=['D1', 'D2', 'D3', 'D4'])
print("Tabel Term Frequency (TF) / Bag-of-Words:\n", df_tf)

# c. Menghitung IDF (menggunakan log10(N/DF))
idf_dict = {term: np.log10(N / df_dict[term]) for term in vocab}
df_df_idf = pd.DataFrame({'DF': df_dict, 'IDF (log10)': idf_dict})
print("\nTabel Document Frequency (DF) & IDF:\n", df_df_idf.T)

# d. Menghitung TF-IDF Manual (TF * IDF)
tfidf_manual = df_tf.copy()
for term in vocab:
    tfidf_manual[term] = tfidf_manual[term] * idf_dict[term]

print("\nMatriks TF-IDF (Manual):\n", tfidf_manual)

=== 1. PERHITUNGAN MANUAL TF-IDF ===

Tabel Term Frequency (TF) / Bag-of-Words:
     buatan  data  jaringan  kecerdasan  kembali  komputer  manusia  sistem  \
D1       0     1         0           0        0         1        0       1   
D2       0     0         1           0        0         1        0       1   
D3       1     0         0           1        0         0        1       0   
D4       0     1         0           0        1         0        0       1   

    temu  
D1     0  
D2     0  
D3     0  
D4     1  

Tabel Document Frequency (DF) & IDF:
               buatan     data  jaringan  kecerdasan  kembali  komputer  \
DF           1.00000  2.00000   1.00000     1.00000  1.00000   2.00000   
IDF (log10)  0.60206  0.30103   0.60206     0.60206  0.60206   0.30103   

             manusia    sistem     temu  
DF           1.00000  3.000000  1.00000  
IDF (log10)  0.60206  0.124939  0.60206  

Matriks TF-IDF (Manual):
      buatan     data  jaringan  kecerdasan  kembali  kompu

In [9]:
# 4 & 5. Implementasi Scikit-Learn dan Tampilkan Matriks[cite: 9]
print("=== 2. PERHITUNGAN TF-IDF SCIKIT-LEARN ===\n")

# Inisialisasi TfidfVectorizer
# Catatan: Scikit-learn secara default menggunakan log natural (ln), smoothing, dan L2-normalization,
# sehingga rentang nilainya akan berbeda dengan perhitungan manual dasar, namun proporsi bobotnya searah.
vectorizer = TfidfVectorizer()
tfidf_sklearn = vectorizer.fit_transform(dokumen_bersih)

# Menampilkan Matriks hasil scikit-learn
df_tfidf_sklearn = pd.DataFrame(
    tfidf_sklearn.toarray(), 
    index=['D1', 'D2', 'D3', 'D4'], 
    columns=vectorizer.get_feature_names_out()
)
print("Matriks TF-IDF (Scikit-Learn):\n", df_tfidf_sklearn)

=== 2. PERHITUNGAN TF-IDF SCIKIT-LEARN ===

Matriks TF-IDF (Scikit-Learn):
      buatan      data  jaringan  kecerdasan  kembali  komputer  manusia  \
D1  0.00000  0.613667  0.000000     0.00000  0.00000  0.613667  0.00000   
D2  0.00000  0.000000  0.702035     0.00000  0.00000  0.553492  0.00000   
D3  0.57735  0.000000  0.000000     0.57735  0.00000  0.000000  0.57735   
D4  0.00000  0.453005  0.000000     0.00000  0.57458  0.000000  0.00000   

      sistem     temu  
D1  0.496816  0.00000  
D2  0.448100  0.00000  
D3  0.000000  0.00000  
D4  0.366747  0.57458  


In [10]:
# 6. Analisis Singkat[cite: 9]
analisis = """
Berdasarkan hasil perhitungan TF-IDF, term yang memiliki bobot tertinggi pada masing-masing dokumen adalah term yang paling spesifik dan unik mewakili topik dokumen tersebut. 
Sebagai contoh, pada D3 (Kecerdasan buatan dapat meniru manusia), term 'buatan', 'kecerdasan', dan 'manusia' mendapatkan bobot TF-IDF maksimal karena term tersebut hanya muncul di D3 (nilai IDF tinggi). 
Sebaliknya, term 'sistem' memiliki bobot lebih rendah pada D1, D2, dan D4 meskipun term tersebut ada, karena kemunculannya di banyak dokumen menurunkan nilai keunikannya (IDF rendah). 
Hal ini menunjukkan bahwa TF-IDF sangat efektif dalam menonjolkan kata kunci penting yang membedakan satu dokumen dengan dokumen lainnya dalam sebuah koleksi.
"""
print("=== ANALISIS SINGKAT ===")
print(analisis)

=== ANALISIS SINGKAT ===

Berdasarkan hasil perhitungan TF-IDF, term yang memiliki bobot tertinggi pada masing-masing dokumen adalah term yang paling spesifik dan unik mewakili topik dokumen tersebut. 
Sebagai contoh, pada D3 (Kecerdasan buatan dapat meniru manusia), term 'buatan', 'kecerdasan', dan 'manusia' mendapatkan bobot TF-IDF maksimal karena term tersebut hanya muncul di D3 (nilai IDF tinggi). 
Sebaliknya, term 'sistem' memiliki bobot lebih rendah pada D1, D2, dan D4 meskipun term tersebut ada, karena kemunculannya di banyak dokumen menurunkan nilai keunikannya (IDF rendah). 
Hal ini menunjukkan bahwa TF-IDF sangat efektif dalam menonjolkan kata kunci penting yang membedakan satu dokumen dengan dokumen lainnya dalam sebuah koleksi.



Berdasarkan hasil perhitungan TF-IDF, term yang memiliki bobot tertinggi pada masing-masing dokumen adalah term yang paling spesifik dan unik mewakili topik dokumen tersebut. 
Sebagai contoh, pada D3 (Kecerdasan buatan dapat meniru manusia), term 'buatan', 'kecerdasan', dan 'manusia' mendapatkan bobot TF-IDF maksimal karena term tersebut hanya muncul di D3 (nilai IDF tinggi). 
Sebaliknya, term 'sistem' memiliki bobot lebih rendah pada D1, D2, dan D4 meskipun term tersebut ada, karena kemunculannya di banyak dokumen menurunkan nilai keunikannya (IDF rendah). 
Hal ini menunjukkan bahwa TF-IDF sangat efektif dalam menonjolkan kata kunci penting yang membedakan satu dokumen dengan dokumen lainnya dalam sebuah koleksi.
